# Model 1 — Panoramic X-ray (OPG) — Training

Detect + FDI-number teeth and detect findings (caries / deep caries / periapical / impacted) on panoramic X-rays.

**Architecture:** YOLO11s (two detectors: teeth+FDI, findings) · **imgsz:** 1024 · **Dataset:** DENTEX

Run on Kaggle (GPU T4, Internet On) or locally with a CUDA GPU. Cells run top to bottom.

## 1. Environment

In [ ]:
!pip -q install "numpy<2.0" "ultralytics==8.3.0"
import torch, ultralytics
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| GPUs", torch.cuda.device_count())
ultralytics.checks()
# On Kaggle: if numpy was upgraded, RESTART THE KERNEL once, then continue from cell 2.

## 2. Dataset
Point `DATA` at this model's `data.yaml` (train/valid/test splits).

In [ ]:
DATA = "data/data.yaml"   # <-- path to data.yaml with train/valid/test
import yaml, glob, os
d = yaml.safe_load(open(DATA))
print("classes:", d.get("names"))
for sp in ("train","valid","test"):
    p = d.get(sp);  n = len(glob.glob(os.path.join(os.path.dirname(DATA), str(p), "*"))) if p else 0
    print(f"{sp}: {p}")

## 3. Train

In [ ]:
# Kaggle note: single GPU avoids the DDP/Ray callback crash on multi-GPU.
import ray.train._internal.session as _s
if not hasattr(_s, "_get_session"): _s._get_session = lambda *a, **k: None

from ultralytics import YOLO
model = YOLO("yolo11s.pt")
results = model.train(
    data=DATA, epochs=100, imgsz=1024, batch=16, device=0,
    patience=25, seed=42, project="runs", name="train", exist_ok=True,
    close_mosaic=10, verbose=True)
print("TRAIN_DONE")

## 4. Validate on the test split

In [ ]:
best = "runs/train/weights/best.pt"
m = YOLO(best)
mt = m.val(data=DATA, split="test", imgsz=1024, device=0)
print("test mAP50:", round(float(mt.box.map50),4), "| mAP50-95:", round(float(mt.box.map),4))
for i, nm in m.names.items():
    print(f"  {nm:12s} mAP50={float(mt.box.ap50[i]):.3f}")

## 5. Export + save weights
Copy `best.pt` into this model's `models/` folder.

In [ ]:
import shutil, os
os.makedirs("models", exist_ok=True)
shutil.copy(best, "models/best.pt")
try:
    m.export(format='onnx', imgsz=1024, opset=12)
    shutil.copy('runs/train/weights/best.onnx', 'models/best.onnx')
except Exception as e:
    print('onnx export skipped:', e)
print("saved -> models/")